# Pipeline vectorisation & matching offre/CV

Ce notebook guide pas à pas la construction d'un pipeline de vectorisation et de matching, avec affichage à chaque étape.

**Plan :**
1. Chargement des ressources (offre test, formations, expériences, projets)
2. Phase 1 : Construction du `cv_vector` (approche simple)
3. Phase 2 : Variante améliorée (3 points d'amélioration)
4. Phase 3 : Matching offre/CV
5. Phase 4 : Ouverture vers des stratégies avancées
6. Préparation des dossiers manquants et structure des CV


## 1. Chargement des ressources (offre test, formations, expériences, projets)

On commence par charger les fichiers JSON présents dans le dossier `data/` pour une offre test et les différentes composantes du CV (formations, expériences, projets). Un affichage est réalisé à chaque étape.

In [33]:

# =======IMPORTS========
import os
import json
import logging
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple

# ========IMPORTS CLASSES========
from cv_pipeline import CV

# Load environment variables from .env file
load_dotenv()

True

In [28]:

# ========CONFIG LOGGING========
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


In [65]:
# ========CHEMINS RESSOURCES========
import os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OFFERS_DIR = os.path.join(DATA_DIR, "offers")
FORMATIONS_DIR = os.path.join(DATA_DIR, "formations")
EXPERIENCES_DIR = os.path.join(DATA_DIR, "experiences")
PROJECTS_DIR = os.path.join(DATA_DIR, "projects")


In [66]:
# ==========INSTANCES========
cv = CV(data_dir=DATA_DIR)

In [78]:
# ========PIPELINE OFFER - CV PROCESS======
formations, experiences, projects = cv.load_alls()


In [79]:
print(f"Formations chargées : {len(formations)}")
print(f"Expériences chargées : {len(experiences)}")
print(f"Projets chargés : {len(projects)}")

Formations chargées : 22
Expériences chargées : 0
Projets chargés : 0


In [80]:
logger.info(f'--- Expériences chargées ({len(experiences)}) ---') 
for e in experiences:
    logger.info(e)
    logger.info('---')

2026-04-18 23:51:24,973 - INFO - --- Expériences chargées (0) ---


---

## 2. Phase 1 : Construction du `cv_vector` (approche simple)

On concatène les textes des formations, expériences et projets pour obtenir une représentation textuelle du CV. Cette base servira à la vectorisation.

Un affichage du texte concaténé est réalisé.

In [35]:
# Construction du texte concaténé du CV (approche simple)
def extract_texts(items, keys=('title', 'description', 'content', 'texte')):
    texts = []
    for item in items:
        for key in keys:
            if key in item:
                texts.append(str(item[key]))
    return texts

cv_text = '\n'.join(
    extract_texts(formations) +
    extract_texts(experiences) +
    extract_texts(projects)
)

print('--- Texte concaténé du CV (approche simple) ---')
print(cv_text)


--- Texte concaténé du CV (approche simple) ---
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).
Parcours terminale scientifique avec forte orientation mathematiques et sciences physiques.
Stage HYPPOCAMPE (theme OXFORD et plus loin) avec travaux de recherche sur automate programmable et animation d'un stand mathematique.
TIPE sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal, avec modelisation dynamique des interactions bulles/onde/calcul.
Mission interimaire orientee relation client en agence bancaire.


---

## 3. Phase 2 : Variante améliorée de l'approche 1

Trois axes d'amélioration :
1. **Pondération des sections** (ex : expériences > formations)
2. **Nettoyage et enrichissement sémantique** (lemmatisation, suppression des stopwords, ajout de mots-clés)
3. **Structuration du texte** (ajout de séparateurs, titres de sections)

Un affichage est réalisé après chaque amélioration.

In [36]:
# 1. Pondération des sections (expériences > formations > projets)
weights = {'formations': 1, 'experiences': 2, 'projects': 2}

def weighted_texts(formations, experiences, projects):
    texts = []
    texts += extract_texts(formations) * weights['formations']
    texts += extract_texts(experiences) * weights['experiences']
    texts += extract_texts(projects) * weights['projects']
    return '\n'.join(texts)

cv_text_weighted = weighted_texts(formations, experiences, projects)
print('--- Texte pondéré du CV ---')
print(cv_text_weighted)


--- Texte pondéré du CV ---
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).
Parcours terminale scientifique avec forte orientation mathematiques et sciences physiques.
Stage HYPPOCAMPE (theme OXFORD et plus loin) avec travaux de recherche sur automate programmable et animation d'un stand mathematique.
TIPE sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal, avec modelisation dynamique des interactions bulles/onde/calcul.
Mission interimaire orientee relation client en agence bancaire.
TIPE sur la localisa

In [37]:
# 2. Nettoyage et enrichissement sémantique (exemple simple)
import re
import spacy
from spacy.lang.fr.stop_words import STOP_WORDS

nlp = spacy.blank('fr')

def clean_and_enrich(text):
    # Nettoyage basique
    text = re.sub(r'[^\w\s]', ' ', text)
    text = text.lower()
    # Lemmatisation et suppression des stopwords
    doc = nlp(text)
    tokens = [t.lemma_ for t in doc if t.text not in STOP_WORDS and len(t.text) > 2]
    # Ajout de mots-clés (exemple)
    keywords = ['python', 'data', 'machine learning']
    tokens += keywords
    return ' '.join(tokens)

cv_text_cleaned = clean_and_enrich(cv_text_weighted)
print('--- Texte nettoyé et enrichi du CV ---')
print(cv_text_cleaned)


--- Texte nettoyé et enrichi du CV ---
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     python data machine learning


In [38]:
# 3. Structuration du texte (ajout de titres de sections)
def structured_cv_text(formations, experiences, projects):
    parts = []
    if formations:
        parts.append('## Formations\n' + '\n'.join(extract_texts(formations)))
    if experiences:
        parts.append('## Expériences\n' + '\n'.join(extract_texts(experiences)))
    if projects:
        parts.append('## Projets\n' + '\n'.join(extract_texts(projects)))
    return '\n\n'.join(parts)

cv_text_structured = structured_cv_text(formations, experiences, projects)
print('--- Texte structuré du CV ---')
print(cv_text_structured)


--- Texte structuré du CV ---
## Formations
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).

## Expériences
Parcours terminale scientifique avec forte orientation mathematiques et sciences physiques.
Stage HYPPOCAMPE (theme OXFORD et plus loin) avec travaux de recherche sur automate programmable et animation d'un stand mathematique.
TIPE sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal, avec modelisation dynamique des interactions bulles/onde/calcul.
Mission interimaire orientee relation client en agenc

---

## 4. Phase 3 : Matching offre/CV

On vectorise l'offre test et le CV, puis on calcule la similarité (cosinus) entre les deux. Affichage du score et des éléments clés.

In [39]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
model.save('./models/all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3564.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.68it/s]


In [40]:
# Vectorisation et matching (exemple avec SentenceTransformer)
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('./models/all-MiniLM-L6-v2')

# Vectorisation
offer_text = offer.get('description', '')
offer_vec = model.encode(offer_text, convert_to_tensor=True)
cv_vec = model.encode(cv_text_structured, convert_to_tensor=True)

# Similarité cosinus
similarity = float(util.cos_sim(offer_vec, cv_vec))
print(f'Similarité offre/CV : {similarity:.3f}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2193.70it/s]


Similarité offre/CV : 0.241


### Analyse du score de similarité : inspection des textes et alignement

In [41]:
# 1. Affichage du texte de l'offre
print('--- Texte de l\'offre ---')
print(offer.get('description', ''))

--- Texte de l'offre ---
La Banque de France recherche un Data Scientist pour traiter, analyser et valoriser des données issues d'enquêtes auprès d'entreprises. Missions : automatisation de la chaîne de traitement, reporting, amélioration méthodologique, participation à des groupes de travail, études statistiques. Compétences : statistiques avancées, data science, outils R/SAS, autonomie, rigueur, esprit d'équipe.


In [42]:
# 2. Affichage du texte structuré du CV
print('--- Texte structuré du CV ---')
print(cv_text_structured)

--- Texte structuré du CV ---
## Formations
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).

## Expériences
Parcours terminale scientifique avec forte orientation mathematiques et sciences physiques.
Stage HYPPOCAMPE (theme OXFORD et plus loin) avec travaux de recherche sur automate programmable et animation d'un stand mathematique.
TIPE sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal, avec modelisation dynamique des interactions bulles/onde/calcul.
Mission interimaire orientee relation client en agenc

In [43]:
# 3. Comparaison des mots-clés entre offre et CV
from collections import Counter
import re

def extract_keywords(text):
    # Extraction simple de mots (à adapter selon besoin)
    words = re.findall(r'\b\w{3,}\b', text.lower())
    return set(words)

offer_keywords = extract_keywords(offer.get('description', ''))
cv_keywords = extract_keywords(cv_text_structured)

common_keywords = offer_keywords & cv_keywords
print(f"Mots-clés communs ({len(common_keywords)}) :", sorted(common_keywords))

Mots-clés communs (7) : ['data', 'des', 'outils', 'pour', 'recherche', 'statistiques', 'traitement']


### Ajustements automatiques du CV à partir de l'offre

In [44]:
# 4. Ajout des mots-clés de l'offre dans le CV (si pertinents)
def enrich_cv_with_offer_keywords(cv_text, offer_keywords):
    # Ajoute les mots-clés de l'offre à la fin du CV (à adapter pour chaque section)
    return cv_text + '\n\nMots-clés de l\'offre : ' + ', '.join(sorted(offer_keywords))

cv_text_enriched = enrich_cv_with_offer_keywords(cv_text_structured, offer_keywords)
print('--- CV enrichi avec mots-clés de l\'offre ---')
print(cv_text_enriched)

--- CV enrichi avec mots-clés de l'offre ---
## Formations
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).

## Expériences
Parcours terminale scientifique avec forte orientation mathematiques et sciences physiques.
Stage HYPPOCAMPE (theme OXFORD et plus loin) avec travaux de recherche sur automate programmable et animation d'un stand mathematique.
TIPE sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal, avec modelisation dynamique des interactions bulles/onde/calcul.
Mission interimaire orientee relation 

In [45]:
# 4bis. Enrichissement automatique des sections du CV avec les mots-clés manquants de l'offre

def enrich_sections_with_missing_keywords(cv_text, offer_keywords, cv_keywords):
    missing_keywords = offer_keywords - cv_keywords
    if not missing_keywords:
        return cv_text  # Rien à ajouter
    
    sections = cv_text.split('## ')
    enriched_sections = []
    for section in sections:
        if not section.strip():
            continue
        # Ajout d'une phrase d'enrichissement si la section n'a pas déjà tous les mots-clés
        section_lower = section.lower()
        missing_in_section = [kw for kw in missing_keywords if kw not in section_lower]
        if missing_in_section:
            phrase = f"\n\nMots-clés recherchés dans l'offre : {', '.join(missing_in_section)}."
            enriched_sections.append('## ' + section.strip() + phrase)
        else:
            enriched_sections.append('## ' + section.strip())
    return '\n\n'.join(enriched_sections)

cv_text_enriched_sections = enrich_sections_with_missing_keywords(cv_text_structured, offer_keywords, cv_keywords)
print('--- CV enrichi par section avec mots-clés manquants ---')
print(cv_text_enriched_sections)

--- CV enrichi par section avec mots-clés manquants ---
## Formations
Parcours d'ingenieur axe sur les mathematiques appliquees, la data, l'informatique et la finance, avec une specialisation progressive vers la finance computationnelle et les systemes d'information financiers.
Formation scientifique generale avec base solide en mathematiques et sciences physiques.
Travaux d'initiative personnelle encadree (TIPE) sur l'utilisation des ondes ultrasonores pour fragmenter un calcul renal.
TIPE sur la localisation des fuites d'eau potable a l'aide d'un capteur acoustique (SePem300).

Mots-clés recherchés dans l'offre : équipe, enquêtes, valoriser, scientist, reporting, sas, rigueur, traiter, avancées, méthodologique, entreprises, compétences, issues, esprit, autonomie, groupes, travail, analyser, amélioration, france, chaîne, auprès, automatisation, banque, études, données, participation, missions.

## Expériences
Parcours terminale scientifique avec forte orientation mathematiques et scie

In [46]:
# 4ter. Nouvelle vectorisation et comparaison des mots-clés après enrichissement contextuel

# Vectorisation
cv_vec_enriched_sections = model.encode(cv_text_enriched_sections, convert_to_tensor=True)
similarity_enriched_sections = float(util.cos_sim(offer_vec, cv_vec_enriched_sections))
print(f"Similarité offre/CV après enrichissement par section : {similarity_enriched_sections:.3f}")

# Comparaison des mots-clés
cv_keywords_enriched_sections = extract_keywords(cv_text_enriched_sections)
common_keywords_enriched_sections = offer_keywords & cv_keywords_enriched_sections
print(f"Mots-clés communs après enrichissement par section ({len(common_keywords_enriched_sections)}) :", sorted(common_keywords_enriched_sections))

Similarité offre/CV après enrichissement par section : 0.387
Mots-clés communs après enrichissement par section (36) : ['amélioration', 'analyser', 'auprès', 'automatisation', 'autonomie', 'avancées', 'banque', 'chaîne', 'compétences', 'data', 'des', 'données', 'enquêtes', 'entreprises', 'esprit', 'france', 'groupes', 'issues', 'missions', 'méthodologique', 'outils', 'participation', 'pour', 'recherche', 'reporting', 'rigueur', 'sas', 'science', 'scientist', 'statistiques', 'traitement', 'traiter', 'travail', 'valoriser', 'équipe', 'études']


In [ ]:
# 5. Filtrage du CV : ne garder que les sections contenant des mots-clés de l'offre
def filter_cv_sections_by_keywords(cv_text, offer_keywords):
    sections = cv_text.split('## ')
    filtered = []
    for section in sections:
        if any(kw in section.lower() for kw in offer_keywords):
            filtered.append('## ' + section if not section.startswith('## ') else section)
    return '\n\n'.join(filtered)

cv_text_filtered = filter_cv_sections_by_keywords(cv_text_structured, offer_keywords)
print('--- CV filtré par mots-clés de l\'offre ---')
print(cv_text_filtered)

In [ ]:
# 6. Structuration améliorée : ajout d'un résumé ciblé
def add_targeted_summary(cv_text, offer_keywords):
    summary = f"Résumé ciblé : Ce CV met en avant les compétences suivantes recherchées dans l'offre : {', '.join(sorted(offer_keywords))}."
    return summary + '\n\n' + cv_text

cv_text_targeted = add_targeted_summary(cv_text_filtered, offer_keywords)
print('--- CV avec résumé ciblé ---')
print(cv_text_targeted)

### Vérification avancée : concepts et mots partagés après ajustement

In [ ]:
# 7. Vérification des mots-clés communs après ajustement
cv_keywords_targeted = extract_keywords(cv_text_targeted)
common_keywords_targeted = offer_keywords & cv_keywords_targeted
print(f"Mots-clés communs après ajustement ({len(common_keywords_targeted)}) :", sorted(common_keywords_targeted))

In [ ]:
# 8. Nouvelle similarité offre/CV après ajustement
cv_vec_targeted = model.encode(cv_text_targeted, convert_to_tensor=True)
similarity_targeted = float(util.cos_sim(offer_vec, cv_vec_targeted))
print(f"Nouvelle similarité offre/CV après ajustement : {similarity_targeted:.3f}")

---

## 5. Phase 4 : Ouverture vers des stratégies avancées

- Génération dynamique de CV personnalisés pour chaque offre
- Passage à des CV multi-colonnes pour ATS souples
- Adaptation automatique à la structure de l'offre (sections, mots-clés)
- Utilisation de modèles plus avancés (LLM, embeddings spécialisés)

Affichage d'un résumé des pistes pour aller plus loin.

In [ ]:
# Résumé des pistes avancées
pistes = [
    "Génération dynamique de CV personnalisés pour chaque offre",
    "CV multi-colonnes pour ATS souples",
    "Adaptation automatique à la structure de l'offre (sections, mots-clés)",
    "Utilisation de modèles plus avancés (LLM, embeddings spécialisés)"
]
print('--- Pistes avancées pour aller plus loin ---')
for p in pistes:
    print('-', p)


---

## 6. Préparation des dossiers manquants et structure des CV

- Création des dossiers `data/cv`, `data/lm`, `data/generations` si absents
- Proposition de structure pour les CV (sections, format JSON/Markdown)
- Rappel : les CV à une colonne pour compatibilité ATS stricts

In [ ]:
# Création des dossiers manquants
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)
        print(f'Dossier créé : {path}')
    else:
        print(f'Dossier déjà existant : {path}')

for sub in ['cv', 'lm', 'generations']:
    ensure_dir(os.path.join(DATA_DIR, sub))


### Exemple de structure JSON pour un CV (1 colonne, ATS friendly)

```json
{
  "id": "cv_charly_2026",
  "nom": "Charly Romy Tanga",
  "sections": [
    {"type": "formations", "items": [...]},
    {"type": "experiences", "items": [...]},
    {"type": "projets", "items": [...]}
  ]
}
```

Chaque section contient une liste d'items (format libre ou structuré selon besoin).